# Perfilado del corpus de `vocab.db`

Análisis de calidad de los datos de entrada del Kindle Vocabulary Builder, previo a
la fase de ingesta (F0).

El objetivo no es describir el archivo, sino **decidir qué trabajo de limpieza hay
que escribir y cuál no merece la pena**. Cada apartado termina en una decisión, y
cada decisión va acompañada del número que la justifica.

Las conclusiones consolidadas viven en `docs/esquema_vocab.md`. Este cuaderno es
el trabajo que las produjo.

## 0. Configuración

Se analizan dos exportaciones del mismo dispositivo, con tres días de diferencia.
La segunda permite comprobar qué cambia entre exportaciones, que es lo que
determina si la reimportación puede ser incremental.

Los archivos contienen el historial de lectura del autor y fragmentos literales de
libros con derechos, así que **no se versionan**. La ruta se lee del entorno.

In [1]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 90)

DATA_DIR = Path(os.environ.get("VOCAB_TEST_DATA", "~/vocab-data")).expanduser()
OLD_DB = DATA_DIR / "vocab.db"
NEW_DB = DATA_DIR / "vocab_new.db"

assert OLD_DB.exists(), f"No se encuentra {OLD_DB}. Define VOCAB_TEST_DATA."


def query(sql: str, db: Path = NEW_DB) -> pd.DataFrame:
    with sqlite3.connect(f"file:{db}?mode=ro", uri=True) as connection:
        return pd.read_sql_query(sql, connection)


print(f"Principal : {NEW_DB.name}")
print(f"Anterior  : {OLD_DB.name}")

Principal : vocab_new.db
Anterior  : vocab.db


## 1. Inventario y relaciones

El archivo es un SQLite corriente. Lo primero es ver qué hay y cómo se conecta.

In [2]:
query("""
    SELECT name, (SELECT COUNT(*) FROM pragma_table_info(name)) AS columnas
    FROM sqlite_master WHERE type = 'table' ORDER BY name
""")

,name,columnas
0,BOOK_INFO,0
1,DICT_INFO,0
2,LOOKUPS,0
3,METADATA,0
4,VERSION,0
5,WORDS,0


In [3]:
for table in ("WORDS", "LOOKUPS", "BOOK_INFO", "DICT_INFO"):
    total = query(f"SELECT COUNT(*) AS n FROM {table}").loc[0, "n"]
    print(f"{table:<12} {total:>6}")

WORDS          1505
LOOKUPS        1690
BOOK_INFO        25
DICT_INFO         2


**No hay ninguna clave foránea declarada.** Un cliente gráfico no dibuja relaciones
entre las tablas, y da la impresión de que las palabras y sus frases no están
vinculadas. Sí lo están, por convención:

```
LOOKUPS.word_key → WORDS.id
LOOKUPS.book_key → BOOK_INFO.id
```

Conviene comprobar que esa convención se cumple antes de confiar en ella.

In [4]:
query("""
    SELECT
      (SELECT COUNT(*) FROM LOOKUPS l
        LEFT JOIN WORDS w ON l.word_key = w.id WHERE w.id IS NULL) AS consultas_huerfanas,
      (SELECT COUNT(*) FROM LOOKUPS l
        LEFT JOIN BOOK_INFO b ON l.book_key = b.id WHERE b.id IS NULL) AS libros_huerfanos,
      (SELECT COUNT(*) FROM WORDS w
        LEFT JOIN LOOKUPS l ON l.word_key = w.id WHERE l.id IS NULL) AS palabras_sin_consulta
""")

,consultas_huerfanas,libros_huerfanos,palabras_sin_consulta
0,0,0,0


> **Decisión.** La integridad se cumple pese a no estar declarada. El importador
> puede usar `JOIN` internos sin perder filas y sin comprobaciones defensivas.

## 2. Perfil general

Volumen y reparto, para saber con qué se trabaja.

In [5]:
query("""
    SELECT
      (SELECT COUNT(*) FROM WORDS)                        AS palabras,
      (SELECT COUNT(*) FROM LOOKUPS)                      AS consultas,
      (SELECT COUNT(*) FROM BOOK_INFO)                    AS libros,
      (SELECT COUNT(*) FROM WORDS WHERE lang = 'en')      AS palabras_en,
      (SELECT COUNT(*) FROM WORDS WHERE lang = 'es')      AS palabras_es,
      (SELECT COUNT(*) FROM LOOKUPS l JOIN WORDS w ON l.word_key = w.id
        WHERE w.lang = 'en')                              AS consultas_en
""")

,palabras,consultas,libros,palabras_en,palabras_es,consultas_en
0,1505,1690,25,913,592,1024


El volumen es el dato que fija la naturaleza del proyecto: **suficiente de sobra
para la aplicación, del todo insuficiente para entrenar nada**. Es un proyecto de
inferencia, orquestación y evaluación, no de entrenamiento.

La v1 cubre solo inglés, así que el corpus de trabajo son las consultas en inglés.

### 2.1 Cuántas veces se consulta cada palabra

Determina si una palabra puede tener varios contextos distintos, y por tanto varios
ejercicios anclados a frases reales diferentes.

In [6]:
distribucion = query("""
    SELECT consultas, COUNT(*) AS palabras FROM (
        SELECT word_key, COUNT(*) AS consultas FROM LOOKUPS GROUP BY word_key
    ) GROUP BY consultas ORDER BY consultas
""")
distribucion

,consultas,palabras
0,1,1357
1,2,121
2,3,19
3,4,7
4,6,1


> **Decisión de modelo.** La relación es uno a muchos: una fila en `entries` por
> palabra, una fila en `contexts` por consulta. La minoría con varios contextos
> obtiene variación de contexto sin necesidad de generar nada.
>
> La programación de repasos (FSRS) va por palabra, no por contexto: si fuera por
> contexto, una palabra con cuatro consultas aparecería cuatro veces el mismo día.

## 3. `WORDS`: los campos y sus trampas

### 3.1 `stem` es una lematización parcial y poco fiable

In [7]:
comparacion = query("SELECT word, stem, lang FROM WORDS")
iguales = (comparacion["word"] == comparacion["stem"]).sum()
print(f"stem idéntico a word: {iguales} de {len(comparacion)}")

comparacion[comparacion["word"] != comparacion["stem"]].sample(10, random_state=0)

stem idéntico a word: 921 de 1505


,word,stem,lang
621,colliding,collide,en
229,pérfidos,pérfido,es
1336,Wounded,wound (),en
472,aventaría,aventar,es
990,Ease,ease,en
261,En,en,es
1320,sage,sage (),en
46,eter,éter,es
142,conniventes,connivente,es
1294,loitering,loiter,en


Donde difieren, la calidad es irregular. Casos concretos que ilustran el problema:

In [8]:
query("""
    SELECT word, stem, lang FROM WORDS
    WHERE word IN ('tribulaciones', 'Hay', 'Salvo', 'fiat', 'Vidas', 'abluciones')
""")

,word,stem,lang
0,fiat,fíat,es
1,Salvo,salvo,es
2,tribulaciones,tribulación,es
3,abluciones,ablución,es
4,Hay,haber,es
5,Vidas,vida,es


`tribulaciones → tribulación` es correcto. `Salvo → salvo` solo baja la
capitalización. Y `fiat → fíat` **añade una tilde que no estaba en el texto**.

> **Decisión.** `stem` se conserva como pista orientativa, pero el lema se deriva
> con spaCy analizando la palabra dentro de su frase. No se usa este campo para
> deduplicar.

### 3.2 La capitalización crea entradas duplicadas

`WORDS.id` tiene la forma `idioma:palabra` y conserva la capitalización original,
así que una palabra consultada al inicio de una frase y otra vez en medio produce
dos filas distintas.

In [9]:
query("""
    SELECT LOWER(word) AS lema_aproximado,
           GROUP_CONCAT(id, '  ·  ') AS ids
    FROM WORDS WHERE lang = 'en'
    GROUP BY LOWER(word) HAVING COUNT(*) > 1
""")

,lema_aproximado,ids
0,a,en:a · en:A
1,biases,en:biases · en:Biases
2,broad,en:broad · en:Broad
3,ease,en:Ease · en:ease
4,hence,en:Hence · en:hence
5,nudge,en:Nudge · en:nudge
6,recall,en:recall · en:Recall
7,strained,en:strained · en:Strained
8,vacuum,en:vacuum · en:VACUUM


In [10]:
resumen = query("""
    SELECT COUNT(*) AS filas, COUNT(DISTINCT LOWER(word)) AS distintas
    FROM WORDS WHERE lang = 'en'
""")
print(f"{resumen.loc[0,'filas']} filas → {resumen.loc[0,'distintas']} palabras distintas "
      f"({resumen.loc[0,'filas'] - resumen.loc[0,'distintas']} duplicados por capitalización)")

913 filas → 904 palabras distintas (9 duplicados por capitalización)


> **Decisión.** Bajar a minúsculas antes de agrupar. Sin esto, la aplicación pediría
> estudiar la misma palabra dos veces.
>
> Este número es la cota superior de entradas tras la ingesta: la lematización con
> spaCy lo reducirá más al unir formas flexionadas (`relied` y `rely`).

### 3.3 `category`: un único caso, no se usa

In [11]:
query("SELECT category, COUNT(*) AS n FROM WORDS GROUP BY category")

,category,n
0,0,1504
1,100,1


In [12]:
query("SELECT id, word, lang FROM WORDS WHERE category != 0")

,id,word,lang
0,en:managed,managed,en


La interpretación probable es *en aprendizaje* frente a *dominada*, marcada desde el
propio dispositivo. Con un solo caso no se puede inferir, y el valor no ha cambiado
entre las dos exportaciones.

> **Decisión.** No se usa. El estado de aprendizaje lo gestiona FSRS dentro de la
> aplicación, así que la columna es prescindible y la cuestión queda cerrada.

## 4. `LOOKUPS`: el activo del proyecto

### 4.1 `pos` no es la categoría gramatical

Error fácil de cometer, porque el nombre lo sugiere.

In [13]:
query("SELECT id, pos, usage FROM LOOKUPS LIMIT 3")[["pos", "usage"]]

,pos,usage
0,AYo4AADGBAAA:1402162,"En este suelo podemos hacer valer los derechos del ciudadano frente al estado, o criti..."
1,AbM4AABKAgAA:1415570,"Su largo aprendizaje en la corte le había enseñado a doblar la cerviz, a rebuscar ansi..."
2,AbY4AABmBAAA:1417154,Su entereza era casi proverbial.


El contenido real es un identificador de posición dentro del libro.

> **Decisión.** La categoría gramatical **no está en la base de datos** y hay que
> derivarla con spaCy analizando la palabra en su frase, lo que además desambigua
> por contexto. Es requisito de la métrica D2.

### 4.2 Cobertura de contexto

La frase real del libro es el material de estudio del proyecto. Si faltara en una
parte significativa de las consultas, el diseño entero cambiaría.

In [14]:
query("""
    SELECT COUNT(*) AS consultas,
           SUM(CASE WHEN usage IS NULL OR TRIM(usage) = '' THEN 1 ELSE 0 END) AS sin_frase,
           ROUND(AVG(LENGTH(usage))) AS longitud_media,
           MAX(LENGTH(usage))        AS longitud_maxima
    FROM LOOKUPS
""")

,consultas,sin_frase,longitud_media,longitud_maxima
0,1690,0,172.0,1003


### 4.3 ¿Aparece la palabra literalmente en su frase?

De esto depende que el ejercicio de hueco sobre la frase original sea una simple
sustitución de cadena o requiera algo más.

In [15]:
pares = query("""
    SELECT w.word, l.usage FROM LOOKUPS l JOIN WORDS w ON l.word_key = w.id
""")
fallos = pares[~pares.apply(lambda r: r["word"] in r["usage"], axis=1)]
print(f"La palabra NO aparece literal en su frase: {len(fallos)} de {len(pares)}")

La palabra NO aparece literal en su frase: 0 de 1690


> **Decisión.** Cobertura del 100 %, comprobada en las dos exportaciones.
> `cloze_original` es una sustitución de cadena trivial y perfectamente fiable,
> **sin ninguna llamada a un modelo de lenguaje**.

## 5. Suciedad de las frases

Es el bloque que determina el trabajo real de la fase de ingesta.

### 5.1 Espacios sobrantes

In [16]:
frases = query("SELECT id, usage FROM LOOKUPS")
con_espacios = (frases["usage"] != frases["usage"].str.strip()).sum()
print(f"Empiezan o terminan con espacio: {con_espacios} de {len(frases)} "
      f"({100 * con_espacios / len(frases):.1f} %)")

Empiezan o terminan con espacio: 1665 de 1690 (98.5 %)


> **Decisión.** `trim()` obligatorio. Afecta a casi todo el corpus.

### 5.2 ¿Están truncadas las frases?

La hipótesis inicial era que el dispositivo recorta el contexto por longitud. Se
comprueba mirando cómo terminan las frases.

In [17]:
CIERRE = (".", "!", "?", "”", "’", "»", '"')

limpias = frases["usage"].str.strip()
terminacion = pd.Series("sin puntuación", index=limpias.index)
terminacion[limpias.str.endswith(".")] = "punto"
terminacion[limpias.str.endswith(("!", "?"))] = "interrogación o exclamación"
terminacion[limpias.str.endswith(("”", "’", "»", '"'))] = "comillas de cierre"

terminacion.value_counts().to_frame("casos")

,casos
punto,1584
comillas de cierre,39
interrogación o exclamación,36
sin puntuación,31


Un 1,8 % no acaba en puntuación reconocible. El recuento por sí solo sugiere
truncamiento. **Hay que mirar los casos concretos**, que es donde aparece el
hallazgo.

In [18]:
sospechosas = limpias[~limpias.str.endswith(CIERRE)]
pd.DataFrame({"final": sospechosas.str[-45:]}).head(15)

,final
5,gún contrapunto surgido de la opinión pública
6,une vraie connaissance de ville d’eau»[59].[
12,ces destacan por su candor y su moderación.»[
17,"1942, fecha en la que descendieron al 78 %.["
38,sta última que no deja de resultar extraña—.[
39,sta última que no deja de resultar extraña—.[
45,nalidad y ávidos de progreso y de renovacione
92,a primera colisión un drama jamás superado.»[
93,donde me sea dado hacerlo razonablemente”.»[
136,que estalló la crisis del 18 Brumario»[102].[


No están truncadas: están **sucias**. La frase llega completa y lo que sobra es el
corchete de apertura de una llamada a nota al pie del libro.

In [19]:
con_corchete = sospechosas.str.endswith("[").sum()
print(f"De las {len(sospechosas)} sin puntuación, terminan en '[': {con_corchete}")
print(f"Candidatas a corte real: {len(sospechosas) - con_corchete} "
      f"({100 * (len(sospechosas) - con_corchete) / len(frases):.1f} % del corpus)")

De las 31 sin puntuación, terminan en '[': 17
Candidatas a corte real: 14 (0.8 % del corpus)


> **Decisión, y reencuadre del trabajo.** El problema no es el truncamiento sino la
> suciedad de maquetación. El trabajo principal es **limpieza de sufijo**; el
> segmentador de frases queda como salvaguarda para la decena de casos restantes,
> no como etapa principal.
>
> Las frases realmente cortadas **no se reconstruyen con un modelo**: el texto que
> falta existe en el libro pero no en la base de datos, así que un modelo solo
> podría fabricarlo. Se marcan y se les asigna otro tipo de ejercicio.

### 5.3 Otras marcas de maquetación

Además del corchete final, conviene buscar referencias intercaladas y caracteres
invisibles antes de dar la limpieza por cerrada.

In [20]:
import re

patrones = {
    "nota al pie intercalada  [59]": r"\[\d+\]",
    "corchete de apertura suelto":   r"\[(?!\d)",
    "espacio no separable  \\xa0":    "\xa0",
    "saltos de línea":               r"[\n\r]",
    "espacios múltiples":            r"\s{2,}",
}
pd.DataFrame(
    [(nombre, frases["usage"].str.contains(patron, regex=True).sum())
     for nombre, patron in patrones.items()],
    columns=["patrón", "casos"],
)

,patrón,casos
0,nota al pie intercalada [59],87
1,corchete de apertura suelto,47
2,espacio no separable \xa0,39
3,saltos de línea,0
4,espacios múltiples,5


In [21]:
ejemplos = frases[frases["usage"].str.contains(r"\[\d+\]", regex=True)]
ejemplos["usage"].str.strip().str[:110].head(5).to_frame("ejemplo")

,ejemplo
0,"En este suelo podemos hacer valer los derechos del ciudadano frente al estado, o criti..."
4,"Al responderle, Leo Amery señaló: «[Churchill] se empeña siempre, y a toda costa, en s..."
6,Clementine resume su devaneo con un toque de ironía y recurriendo «a un dicho que pare...
8,"Fueron versos que no recitó en el debate pero que sí que habría de traer a colación, y..."
14,"No le di la oportunidad de estrecharnos las manos, pero desde luego le sometí a un lar..."


> **Decisión.** La limpieza determinista debe cubrir, en este orden: eliminar
> referencias `[n]`, sustituir espacios no separables, colapsar espacios múltiples,
> recortar el corchete final y aplicar `trim()`.
>
> **Se ejecuta antes de spaCy.** Un segmentador que recibe `…more.[59].[` detecta
> dos oraciones donde hay una, y eso degrada tanto la segmentación como el análisis
> de dependencias del que dependen las métricas D8 y D9.

### 5.4 Ruido por toques accidentales

Algunas entradas no son vocabulario: son pulsaciones involuntarias sobre palabras
funcionales al pasar página.

In [22]:
query("""
    SELECT w.word, w.lang, COUNT(*) AS consultas
    FROM WORDS w JOIN LOOKUPS l ON l.word_key = w.id
    WHERE LENGTH(w.word) <= 6
    GROUP BY w.id ORDER BY LENGTH(w.word) LIMIT 15
""")

,word,lang,consultas
0,A,en,1
1,M,en,2
2,a,en,2
3,D,es,1
4,F,es,1
5,a,es,2
6,y,es,1
7,Do,en,1
8,He,en,1
9,am,en,1


> **Decisión.** Hay que filtrarlas. La longitud no es un buen criterio —hay palabras
> cortas perfectamente legítimas—, así que el filtro se apoya en la categoría
> gramatical que devuelve spaCy en contexto, no en una lista fija por idioma.
>
> Queda por medir cuántas entradas legítimas se pierden con esa regla, antes de
> darla por buena.

## 6. Qué cambia entre exportaciones

El dispositivo acumula: cada exportación contiene lo anterior más lo nuevo. De este
apartado depende que la reimportación pueda ser incremental sin destruir el progreso
del usuario.

In [23]:
def conjunto(tabla: str, db: Path) -> set:
    return set(query(f"SELECT id FROM {tabla}", db)["id"])


filas = []
for tabla in ("LOOKUPS", "WORDS", "BOOK_INFO"):
    antiguo, nuevo = conjunto(tabla, OLD_DB), conjunto(tabla, NEW_DB)
    filas.append({
        "tabla": tabla,
        "en ambas": len(antiguo & nuevo),
        "desaparecidas": len(antiguo - nuevo),
        "añadidas": len(nuevo - antiguo),
    })
pd.DataFrame(filas)

,tabla,en ambas,desaparecidas,añadidas
0,LOOKUPS,1511,0,179
1,WORDS,1345,0,160
2,BOOK_INFO,25,0,0


Ninguna fila desaparece. Pero que el identificador sobreviva no garantiza que el
contenido sea el mismo: hay que comparar todos los campos de las filas comunes.

In [24]:
def comparar(tabla: str, columnas: str) -> pd.DataFrame:
    antiguo = query(f"SELECT {columnas} FROM {tabla}", OLD_DB).set_index("id")
    nuevo = query(f"SELECT {columnas} FROM {tabla}", NEW_DB).set_index("id")
    comunes = antiguo.index.intersection(nuevo.index)
    distintas = antiguo.loc[comunes].compare(nuevo.loc[comunes])
    return distintas


for tabla, columnas in [
    ("LOOKUPS", "id, word_key, book_key, pos, usage, timestamp"),
    ("BOOK_INFO", "id, lang, title, authors"),
    ("WORDS", "id, word, stem, lang, category, timestamp"),
]:
    distintas = comparar(tabla, columnas)
    campos = list(distintas.columns.get_level_values(0).unique())
    print(f"{tabla:<10} filas con algún cambio: {len(distintas):>3}"
          f"{'   campos: ' + ', '.join(campos) if campos else ''}")

LOOKUPS    filas con algún cambio:   0
BOOK_INFO  filas con algún cambio:   0
WORDS      filas con algún cambio:  16   campos: timestamp


`LOOKUPS` y `BOOK_INFO` son idénticas campo a campo. En `WORDS` cambia un solo
campo, y conviene entender por qué antes de darlo por inocuo.

In [25]:
cambiadas = comparar("WORDS", "id, word, stem, lang, category, timestamp").index

con_consulta_nueva = conjunto("LOOKUPS", NEW_DB) - conjunto("LOOKUPS", OLD_DB)
palabras_reconsultadas = set(
    query("SELECT id, word_key FROM LOOKUPS", NEW_DB)
    .set_index("id").loc[list(con_consulta_nueva), "word_key"]
)

print(f"Palabras con timestamp modificado: {len(cambiadas)}")
print(f"¿Todas tienen una consulta nueva? {set(cambiadas) <= palabras_reconsultadas}")

Palabras con timestamp modificado: 16
¿Todas tienen una consulta nueva? True


> **Decisión.** `WORDS.timestamp` es la fecha de la **última** consulta, no de la
> primera. `entries.first_seen_at` se deriva de `MIN(LOOKUPS.timestamp)` de las
> consultas de esa palabra; usar `WORDS.timestamp` guardaría la fecha más reciente
> bajo un nombre que promete lo contrario.
>
> **Consecuencia principal.** Los identificadores son estables y las consultas
> pasadas nunca se reescriben, así que `LOOKUPS.id` y `WORDS.id` sirven como
> `external_id` para deduplicar. La ingesta puede insertar solo lo que no existe
> y no tocar nunca las tablas donde vive el progreso del usuario.

## 7. Resumen de las decisiones de ingesta

| # | Hallazgo | Decisión | Coste |
|---|---|---|---|
| 1 | Sin claves foráneas, pero integridad correcta | `JOIN` internos, sin comprobaciones defensivas | — |
| 2 | `stem` es lematización parcial y a veces errónea | Derivar el lema con spaCy en contexto | Medio |
| 3 | La capitalización duplica entradas | Bajar a minúsculas antes de agrupar | Bajo |
| 4 | `category` tiene un único caso distinto de 0 | No se usa; FSRS gestiona el estado | — |
| 5 | `pos` es un identificador de posición | Derivar la categoría gramatical con spaCy | Medio |
| 6 | La palabra aparece literal en su frase el 100 % | `cloze_original` sin modelo de lenguaje | — |
| 7 | Espacios sobrantes en el 98,5 % | `trim()` obligatorio | Bajo |
| 8 | Notas al pie `[n]` y corchete final | Limpieza de sufijo **antes** de spaCy | Bajo |
| 9 | Frases realmente cortadas: ~0,7 % | Marcar, nunca reconstruir con un modelo | Bajo |
| 10 | Toques accidentales sobre palabras funcionales | Filtrar por categoría gramatical | Medio |
| 11 | Identificadores estables entre exportaciones | Ingesta incremental por `external_id` | Medio |

**Lo que este análisis descarta.** Ninguna de las once decisiones requiere un modelo
de lenguaje. Toda la fase de ingesta es determinista, y eso no es una carencia sino
el resultado de haber medido antes de escribir código.